[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/02_%E7%B5%B1%E8%A8%88_4%E7%B4%9A/05_PPDAC%E3%81%A8%E3%82%B0%E3%83%A9%E3%83%95%E3%81%AE%E8%AA%AD%E3%81%BF%E6%96%B9.ipynb)

> Colabで実行可。最初に「① セットアップ」セルを実行（Colabはデータを自動生成、ローカルは何もしない）。

In [ ]:
# === ① セットアップ（最初に実行してください）===============================
# Colabなどデータが無い環境では、教材と同一の合成データを自動生成します。
# ローカル/Codespacesで data/ がある場合は何もしません（再現性のため乱数seedは固定）。
import os
if not os.path.exists('../data/students_scores.csv'):   # データが無い環境（Colab等）か判定
    os.makedirs('/content/nb', exist_ok=True); os.chdir('/content/nb')  # ../data を使えるよう作業場所を作る
    os.makedirs('../data', exist_ok=True)               # データ保存先フォルダを作る
    import numpy as np, pandas as pd                     # 数値計算と表データのライブラリ
    D = '../data'; rng = np.random.default_rng(42)       # 保存先と、結果を固定する乱数生成器
    # --- 生徒120人の成績データを作る ---
    n = 120; cls = rng.choice(['A','B','C'], n)          # 人数とクラス
    math = np.clip(rng.normal(65,15,n),0,100).round().astype(int)        # 数学（0〜100点）
    eng = np.clip(0.6*math+rng.normal(28,10,n),0,100).round().astype(int) # 英語（数学と相関）
    jp = np.clip(rng.normal(70,12,n),0,100).round().astype(int)          # 国語
    hrs = np.clip(rng.normal(1.5,0.8,n)+math/100,0,None).round(1)        # 勉強時間
    pd.DataFrame({'生徒ID':[f'S{i:03d}' for i in range(1,n+1)],'クラス':cls,
        '数学':math,'英語':eng,'国語':jp,'勉強時間':hrs}).to_csv(f'{D}/students_scores.csv',index=False,encoding='utf-8-sig')
    # --- 30日間の天気データ ---
    days = pd.date_range('2026-06-01', periods=30)       # 6月の30日分の日付
    temp = (24+4*np.sin(np.arange(30)/5)+rng.normal(0,2,30)).round(1)    # 気温
    rain = np.clip(rng.gamma(1.2,4,30)*(rng.random(30)<0.4),0,None).round(1)  # 降水量
    pd.DataFrame({'日付':days.strftime('%Y-%m-%d'),'気温':temp,'降水量':rain}).to_csv(f'{D}/weather.csv',index=False,encoding='utf-8-sig')
    # --- BtoB商談400件 ---
    m = 400; inds = rng.choice(['製造','IT','小売','医療','金融','建設'],m,p=[.25,.2,.18,.12,.15,.1])  # 業界
    chs = ['展示会','Web広告','メルマガ','紹介','テレアポ']; ch = rng.choice(chs,m,p=[.2,.3,.2,.15,.15])  # 獲得チャネル
    deal = (rng.lognormal(13,0.7,m)).round(-3).astype(int)   # 商談金額
    wr = {'展示会':.35,'Web広告':.18,'メルマガ':.12,'紹介':.45,'テレアポ':.15}  # チャネル別の受注率
    won = np.array([rng.random()<wr[c] for c in ch])         # 受注したか（チャネルで確率を変える）
    dt = pd.to_datetime('2026-01-01')+pd.to_timedelta(rng.integers(0,180,m),unit='D')  # 受注日
    pd.DataFrame({'商談ID':[f'D{i:04d}' for i in range(1,m+1)],'受注日':dt.strftime('%Y-%m-%d'),
        '業界':inds,'獲得チャネル':ch,'商談金額':deal,'担当者':rng.choice(['田中','鈴木','佐藤','高橋'],m),
        '受注':won.astype(int)}).to_csv(f'{D}/sales_btob.csv',index=False,encoding='utf-8-sig')
    # --- 月×チャネルのマーケ実績 ---
    months = pd.date_range('2026-01-01',periods=6,freq='MS').strftime('%Y-%m')  # 6か月分
    base = {'展示会':2000,'Web広告':12000,'メルマガ':6000,'紹介':800,'テレアポ':1500}  # 表示回数の目安
    ctr = {'展示会':.08,'Web広告':.03,'メルマガ':.05,'紹介':.2,'テレアポ':.12}   # クリック率
    cvr = {'展示会':.12,'Web広告':.04,'メルマガ':.06,'紹介':.25,'テレアポ':.09}   # 獲得率
    cpc = {'展示会':30,'Web広告':80,'メルマガ':5,'紹介':0,'テレアポ':200}        # クリック単価
    rows = []
    for mo in months:                                    # 月ごと
        for c in chs:                                    # チャネルごとに実績を作る
            imp = int(base[c]*rng.uniform(0.8,1.3)); clk = int(imp*ctr[c]*rng.uniform(0.8,1.2))
            cv = int(clk*cvr[c]*rng.uniform(0.7,1.3)); cost = int(clk*cpc[c]*rng.uniform(0.9,1.1))
            rows.append([mo,c,imp,clk,cv,cost])
    pd.DataFrame(rows,columns=['月','チャネル','表示回数','クリック数','獲得数','費用']).to_csv(f'{D}/web_marketing.csv',index=False,encoding='utf-8-sig')
    # --- A/Bテストの申込ログ ---
    ab = []
    for grp,p,size in [('A_青ボタン',0.082,2400),('B_緑ボタン',0.104,2380)]:  # 2群の申込率と人数
        for c in (rng.random(size)<p): ab.append([grp,int(c)])   # 各訪問が申込んだか
    pd.DataFrame(ab,columns=['グループ','申込']).sample(frac=1,random_state=1).reset_index(drop=True).to_csv(f'{D}/ab_test.csv',index=False,encoding='utf-8-sig')
    print('✅ 合成データを生成しました（Colab環境）')
else:
    print('✅ data/ を検出しました（ローカル/Codespaces）')

# 統計4級-05. 統計的問題解決(PPDAC)と基本グラフの読み方

**できるようになること**
- PPDACで問題解決の流れを説明できる
- 目的に合うグラフを選び、正しく読める
- パレート図・幹葉図を作れる

**前提**：`01_Python基礎`　/　**所要**：約25分　/　**レベル**：統計検定4級相当

In [ ]:
import numpy as np               # 数値計算
import pandas as pd              # 表データ
import matplotlib.pyplot as plt  # グラフ描画
plt.rcParams['axes.unicode_minus'] = False       # マイナス記号の文字化けを防ぐ

## 1. PPDAC サイクル

統計で問題を解くときの流れです。頭文字をとってPPDAC。

| 段階 | 意味 | 例（自販機の売上を伸ばしたい） |
|---|---|---|
| **P**roblem | 問題を決める | どの商品がいつ売れる？ |
| **P**lan | 計画する | 1か月、時間帯別に記録しよう |
| **D**ata | データを集める | レジ記録を集計 |
| **A**nalysis | 分析する | グラフ化・平均を比較 |
| **C**onclusion | 結論を出す | 夏の午後は冷たい飲料を増やす |

> 結論からまた新しいProblemが生まれ、サイクルは回り続けます。

## 2. 基本グラフの使い分け

| グラフ | 得意なこと | 使う場面 |
|---|---|---|
| 棒グラフ | 量の大小を比べる | クラス別の人数 |
| 折れ線グラフ | 変化（時間）を見る | 毎月の気温 |
| 円グラフ | 割合（構成比）を見る | 好きな教科の割合 |
| 帯グラフ | 割合を複数で比べる | 年ごとの構成比の変化 |
| 積み上げ棒 | 量と内訳を同時に | 店舗別×商品別売上 |
| パレート図 | 「重要な少数」を見る | 上位で大半を占める要因 |

### 棒グラフ・折れ線・円グラフを描いてみる

In [ ]:
kyoka = pd.Series({'数学':8,'英語':12,'国語':10,'理科':6,'社会':4}, name='人数')  # 教科ごとの人数

fig, ax = plt.subplots(1, 3, figsize=(13,4))   # 1行3列のグラフ領域
kyoka.plot(kind='bar', ax=ax[0], title='棒：好きな教科の人数')   # 左：棒グラフ
ax[0].tick_params(axis='x', rotation=0)        # x軸ラベルを水平に
# 中：折れ線（月ごとの気温）
pd.Series([22,24,28,30,27,25],
          index=['1月','2月','3月','4月','5月','6月']).plot(
          marker='o', ax=ax[1], title='折れ線：気温の変化')
kyoka.plot(kind='pie', ax=ax[2], autopct='%1.0f%%', title='円：割合')  # 右：円グラフ
ax[2].set_ylabel('')                           # 円グラフのy軸ラベルを消す
plt.tight_layout(); plt.show()                 # レイアウトを整えて表示

### 帯グラフ（割合を複数で比較）

In [ ]:
# 端末利用の構成比（2年分）を作る
df_taikei = pd.DataFrame({
    '2020年':[40,35,25], '2026年':[55,30,15]
}, index=['スマホ','PC','その他']).T
ratio = df_taikei.div(df_taikei.sum(axis=1), axis=0) * 100  # 各年で合計100%に変換
ratio.plot(kind='barh', stacked=True, figsize=(8,3),       # 横向き積み上げ＝帯グラフ
           title='帯グラフ：利用端末の割合の変化(%)')
plt.xlabel('％'); plt.show()

### 本物のデータで帯グラフ（e-Stat 社会生活基本調査・2006年）

帯グラフは**複数グループの「構成比」を見比べる**のが得意です。社会生活基本調査から「スポーツ種類別の行動者数（男女別）」を取得し、男女で人気種目の内訳がどう違うかを帯グラフで比べます。

In [ ]:
import os, pandas as pd

def load_sports():
    """社会生活基本調査（2006年・全国）スポーツ種類別×男女の行動者数(千人)。
    ① data/ キャッシュ → ② e-Stat API（要 ESTAT_APP_ID）→ ③ 同梱の主要種目（抜粋）。"""
    path='../data/estat_sports_2006.csv'
    if os.path.exists(path):
        return pd.read_csv(path)
    appid=os.environ.get('ESTAT_APP_ID')
    if appid:
        import requests
        r=requests.get('https://api.e-stat.go.jp/rest/3.0/app/json/getStatsData',
            params={'appId':appid,'statsDataId':'0000092990','cdCat01':'001','cdCat04':'001'},timeout=60)
        j=r.json()['GET_STATS_DATA']
        cls={co['@id']:{c['@code']:c['@name'] for c in (co['CLASS'] if isinstance(co['CLASS'],list) else [co['CLASS']])}
             for co in j['STATISTICAL_DATA']['CLASS_INF']['CLASS_OBJ']}
        rows=[]
        for v in j['STATISTICAL_DATA']['DATA_INF']['VALUE']:
            try: val=float(v['$'])
            except: continue
            rows.append((cls['cat02'][v['@cat02']], cls['cat03'][v['@cat03']], val))
        df=pd.DataFrame(rows,columns=['男女','スポーツ','行動者数_千人'])
        df=df[(df['男女']!='総数') & (df['スポーツ']!='総数')]
        df['スポーツ']=df['スポーツ'].str.replace('（キャッチボールを含む）','',regex=False)
        os.makedirs('../data',exist_ok=True); df.to_csv(path,index=False,encoding='utf-8-sig')
        return df
    male={'ウォーキング・軽い体操':16929,'ボウリング':11637,'つり':8867,'水泳':8226,'野球':8037,
          'ジョギング・マラソン':6523,'ゴルフ(練習場を含む)':8390,'器具を使ったトレーニング':6569,
          'サッカー':5734,'テニス':3249,'バレーボール':2470,'バドミントン':2930}
    female={'ウォーキング・軽い体操':22750,'ボウリング':9500,'つり':2473,'水泳':7462,'野球':1691,
            'ジョギング・マラソン':3443,'ゴルフ(練習場を含む)':1749,'器具を使ったトレーニング':6211,
            'サッカー':1098,'テニス':3069,'バレーボール':3572,'バドミントン':4433}
    rows=[('男',k,v) for k,v in male.items()]+[('女',k,v) for k,v in female.items()]
    return pd.DataFrame(rows,columns=['男女','スポーツ','行動者数_千人'])

sports = load_sports()
sports.head()

In [ ]:
keep=['ウォーキング・軽い体操','ボウリング','つり','水泳','野球','ゴルフ(練習場を含む)',
      '器具を使ったトレーニング','ジョギング・マラソン','サッカー','テニス','バレーボール','バドミントン']
piv=(sports[sports['スポーツ'].isin(keep)]
     .pivot_table(index='男女',columns='スポーツ',values='行動者数_千人',aggfunc='sum')
     .reindex(index=['男','女'],columns=keep))
comp=piv.div(piv.sum(axis=1),axis=0)*100          # 各性別での構成比（%）に直す
ax=comp.plot(kind='barh',stacked=True,figsize=(11,3.2),width=0.6,colormap='tab20')
ax.set_xlabel('構成比（%）'); ax.set_xlim(0,100)
ax.set_title('スポーツ行動者の構成比：男女の比較（2006年・主要種目）')
ax.legend(bbox_to_anchor=(1.005,1.0),fontsize=8,title='種目')
import matplotlib.pyplot as plt; plt.tight_layout(); plt.show()

> 帯グラフにすると、人数の多少ではなく**構成の違い**が読めます。例：男性は「つり・野球・ゴルフ」の比率が高め、女性は「ウォーキング・バドミントン・バレー」の比率が高め。同じ「割合の比較」でも、円グラフを2つ並べるより帯グラフを縦に並べたほうが差を追いやすいです。

### パレート図（重要な少数を見つける）
棒（大きい順）＋累積割合の折れ線。上位2〜3項目で全体の大半を占めることを示します。

In [ ]:
claims = pd.Series({'配送遅延':52,'破損':28,'数量違い':12,'その他':8})  # クレーム件数
claims = claims.sort_values(ascending=False)   # 多い順に並べ替え
cum = claims.cumsum() / claims.sum() * 100     # 累積の割合(%)

fig, ax1 = plt.subplots(figsize=(7,4))         # グラフ領域
ax1.bar(claims.index, claims.values)           # 棒：件数
ax2 = ax1.twinx()                              # 右側にもう1つy軸を追加
ax2.plot(claims.index, cum.values, color='red', marker='o')  # 折れ線：累積割合
ax2.set_ylim(0,110); ax2.set_ylabel('累積％')
ax1.set_ylabel('件数'); plt.title('パレート図：クレームの内訳')
plt.show()
print('上位2項目で全体の', round(cum.iloc[1],1), '%')

## 3. 幹葉図（みきはず）

数を「十の位（幹）」と「一の位（葉）」に分けて並べる図。
ヒストグラムに似ていますが、**元の数字が残る**のが特徴です。

In [ ]:
# 幹葉図を作る関数：十の位を「幹」、一の位を「葉」として並べる
def stem_leaf(data):
    data = sorted(data)                        # 小さい順に並べる
    stems = {}                                 # 幹ごとに葉を貯める入れ物
    for v in data:
        stems.setdefault(v // 10, []).append(v % 10)  # 十の位で分け、一の位を葉に追加
    for s in range(min(stems), max(stems) + 1):       # 幹を小さい順に
        leaves = ''.join(str(x) for x in stems.get(s, []))  # 葉を並べた文字列
        print(f'{s} | {leaves}')               # 「幹 | 葉」で表示

scores = [62,65,67,71,73,73,78,81,82,85,88,90,52,59]  # 点数データ
stem_leaf(scores)                              # 幹葉図を表示
print('読み方: 7|1338 は 71,73,73,78 点の4人')

> **よくある間違い**：円グラフは項目が多い／差が小さいと読みにくい。比較が目的なら棒グラフが無難。3D円グラフは面積が歪むので避けましょう。

## 確認テスト（自動採点）

`ans = None` を**自分の計算で置き換えて実行**すると、その場で正誤が出ます（答えは表示されません）。

In [ ]:
# 採点用の関数を実行（答え合わせに使うだけ）
def _check(name, got, want, tol=1e-2):
    if got is None:
        print(f'⬜ {name}: まだ入力されていません（ans を埋めてね）'); return
    ok = abs(got - want) <= tol
    print(('✅ 正解！ ' if ok else '❌ もう一度 ') + f'{name}: あなたの答え = {got}')

In [ ]:
import pandas as pd
c = pd.Series({'A':60, 'B':25, 'C':10, 'D':5})
# Q1: 最大項目Aの構成比(%)を ans に
ans = None   # 60 / 合計 * 100
_check('Q1 構成比%', ans, 60/c.sum()*100)

In [ ]:
import pandas as pd
c = pd.Series({'A':60, 'B':25, 'C':10, 'D':5})
# Q2: 上位2項目(A,B)の累積構成比(%)を ans に
ans = None
_check('Q2 累積%', ans, (60+25)/c.sum()*100)

---
## 練習問題

**問1.** 次のデータに最も適したグラフを選べ：
(a) 1年間の毎月の売上の推移 (b) アンケートの賛成・反対・どちらでもないの割合
(c) 5つの店舗の来客数の比較

**問2.** `students_scores.csv` の `数学` を幹葉図にしよう。

**問3.** ある店の不具合原因 `{部品A:60, 部品B:25, 部品C:10, その他:5}` でパレート図を描き、
「上位いくつの原因で全体の8割を超えるか」を答えよう。

**問4.** `load_sports()` のデータで、**男女それぞれ行動者数が多い上位5種目**を求めよう。さらに「つり」の行動者数が男女でどれくらい違うか比べてみよう。

In [ ]:
# 問2
import pandas as pd                            # pandasを読み込む
df = pd.read_csv('../data/students_scores.csv')  # データを読み込む

### 使うデータ：`students_scores.csv`（架空の生徒120人の成績）
1行＝生徒1人。数学・英語・国語は0〜100点、勉強時間は1日あたりの時間です。

| 生徒ID | クラス | 数学 | 英語 | 国語 | 勉強時間 |
|---|---|---|---|---|---|
| S001 | A | 52 | 58 | 49 | 2.0 |
| S002 | C | 80 | 79 | 74 | 3.4 |
| S003 | B | 40 | 65 | 91 | 2.0 |

（下のセルの `df.head()` で先頭5行を実際に確認できます）

In [ ]:
# 問3


> **解答例は別ページにまとめています**（ネタバレ防止）。
> 自分で解いてから **[05_PPDACとグラフの読み方 の解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/02_%E7%B5%B1%E8%A8%88_4%E7%B4%9A/05_PPDAC%E3%81%A8%E3%82%B0%E3%83%A9%E3%83%95%E3%81%AE%E8%AA%AD%E3%81%BF%E6%96%B9.md)**

## 用語集 ＆ チートシート

| 用語 | 使う場面 |
|---|---|
| PPDAC | 問題→計画→データ→分析→結論 |
| 棒グラフ | 量の大小を比べる |
| 折れ線 | 時間の変化を見る |
| 円・帯 | 割合（構成比）を見る |
| パレート図 | 重要な少数を見つける |
| 幹葉図 | 元の数字を残して分布を見る |

In [ ]:
# チートシート（グラフ）
import pandas as pd
s = pd.Series({'A':8, 'B':12, 'C':10})
s.plot(kind='bar')      # 棒
s.plot(kind='line')     # 折れ線
s.plot(kind='pie')      # 円
s.cumsum()/s.sum()*100  # 累積構成比（パレート）